In [1]:
import pandas as pd
import re
import emoji
import html
import nltk
from tqdm import tqdm
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

nltk.download('punkt')
nltk.download('punkt_tab')   # ← wajib untuk NLTK >= 3.8
nltk.download('stopwords')

tqdm.pandas()
print("✅ Library berhasil diimport")

✅ Library berhasil diimport


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [2]:
df = pd.read_csv("rawData/dataX.csv")
df = df[['created_at', 'user_id_str', 'full_text']]
df['full_text'] = df['full_text'].fillna('')
df.drop_duplicates(subset=['full_text'], inplace=True)
print(f"Data loaded: {len(df)} rows")

Data loaded: 1882 rows


In [3]:
def clean_twitter(text):
    """
    Pembersihan untuk data Twitter:
    - URL, angka, tanda baca
    - Mention (@username)
    - Hashtag (#) -> kata dipertahankan
    - RT (retweet indicator)
    - Emoji dikonversi ke teks
    - HTML entities
    """
    if not isinstance(text, str):
        return ""
    
    # 1. HTML entities
    text = html.unescape(text)
    
    # 2. Konversi emoji ke teks (contoh :smile: → smile)
    text = emoji.demojize(text, language='en')
    text = re.sub(r':(\w+):', r'\1', text)
    
    # 3. Hapus URL
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    
    # 4. Hapus mention (@username)
    text = re.sub(r'@\w+', '', text)
    
    # 5. Hapus simbol # tapi pertahankan kata (hashtag)
    text = re.sub(r'#(\w+)', r' \1', text)
    
    # 6. Hapus RT (retweet) indicator
    text = re.sub(r'^RT[\s]+', '', text)
    
    # 7. Hapus angka (opsional, untuk LDA biasanya tidak diperlukan)
    text = re.sub(r'\d+', '', text)
    
    # 8. Hapus tanda baca (karakter non-alfanumerik kecuali spasi)
    text = re.sub(r'[^\w\s]', ' ', text)
    
    # 9. Whitespace berlebih
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

In [4]:
def case_folding(text):
    return text.lower()

In [5]:
# Load kamus alay
indo_slang_df = pd.read_csv(
    "https://raw.githubusercontent.com/nasalsabila/kamus-alay/master/colloquial-indonesian-lexicon.csv"
)

# Konversi ke dict — ambil formal pertama jika ada duplikat slang
slang_dict = (
    indo_slang_df
    .drop_duplicates(subset='slang', keep='first')
    .set_index('slang')['formal']
    .to_dict()
)

print(f"Kamus alay dimuat: {len(slang_dict)} entri")

normalization_dict = {
    'yg': 'yang', 'dg': 'dengan', 'dgn': 'dengan', 'ga': 'tidak', 'gak': 'tidak',
    'tdk': 'tidak', 'krn': 'karena', 'sdh': 'sudah', 'udh': 'sudah', 'bs': 'bisa',
    'tp': 'tapi', 'utk': 'untuk', 'dlm': 'dalam', 'pd': 'pada', 'dll': 'dan lain lain',
    'dkk': 'dan kawan kawan', 'rp': 'rupiah',
    'banget': 'sangat', 'bgt': 'sangat', 'deh': '', 'sih': '', 'dong': '', 'kok': '',
    'gw': 'saya', 'gue': 'saya', 'lu': 'kamu', 'loe': 'kamu', 'elo': 'kamu',
    'nyari': 'cari', 'nggak': 'tidak', 'ngga': 'tidak', 'kayak': 'seperti',
    'btw': 'omong omong', 'omg': 'astaga', 'lol': ''
}
combined_dict = {**slang_dict, **normalization_dict}
print(f"Total entri gabungan: {len(combined_dict)}")

def normalize_text(text):
    if not isinstance(text, str):
        return ""
    words = text.split()
    result = []
    for word in words:
        replaced = combined_dict.get(word, word)
        replaced = re.sub(r'^(\w+)-\1$', r'\1', replaced)  # hapus kata berulang
        if replaced.strip():
            result.append(replaced)
    return ' '.join(result)

Kamus alay dimuat: 4331 entri
Total entri gabungan: 4345


In [6]:
def tokenize_text(text):
    """Memecah teks menjadi token kata"""
    return word_tokenize(text)

In [7]:
from Sastrawi.StopWordRemover.StopWordRemoverFactory import (
    StopWordRemoverFactory, StopWordRemover, ArrayDictionary
)

sastrawi_stopwords = set(StopWordRemoverFactory().get_stop_words())

# Tambahan stopword umum yang tidak bermakna
additional_stopwords = {
    'yuk', 'nih', 'deh', 'sih', 'dong', 'kok', 'loh', 'lah',
    'btw', 'omg', 'wkwk', 'haha', 'hehe', 'hihi',
    'via', 'cc', 'rt', 'dm', 'ot',           # artefak Twitter
    'thread', 'twit', 'tweet', 'retweet',
    'like', 'share', 'follow', 'subscribe',
    'http', 'https', 'pic', 'com',            # sisa URL yang lolos
}

keep_words = {
    'korupsi', 'uang', 'negara', 'proyek', 'lelang', 'pengadaan',
    'tidak', 'bukan',    # negasi penting untuk sentimen/topik
}

final_stopwords = (sastrawi_stopwords | additional_stopwords) - keep_words

print(f"Sastrawi stopwords  : {len(sastrawi_stopwords)}")
print(f"Setelah penambahan  : {len(sastrawi_stopwords | additional_stopwords)}")
print(f"Setelah pengurangan : {len(final_stopwords)}")

for word in keep_words:
    status = "AMAN" if word not in final_stopwords else "MASALAH"
    print(f"  '{word}': {status}")

def remove_stopwords(tokens):
    return [w for w in tokens if w not in final_stopwords and len(w) > 2]

Sastrawi stopwords  : 809
Setelah penambahan  : 837
Setelah pengurangan : 835
  'bukan': AMAN
  'korupsi': AMAN
  'negara': AMAN
  'proyek': AMAN
  'pengadaan': AMAN
  'tidak': AMAN
  'lelang': AMAN
  'uang': AMAN


In [8]:
factory = StemmerFactory()
stemmer = factory.create_stemmer()

def stem_tokens(tokens):
    """Stem setiap token ke bentuk dasar"""
    return [stemmer.stem(w) for w in tokens]

In [9]:
print("Memulai preprocessing pipeline...")

# Langkah 1: Data Cleaning
df['cleaned'] = df['full_text'].progress_apply(clean_twitter)

# Langkah 2: Case Folding
df['casefolded'] = df['cleaned'].progress_apply(case_folding)

# Langkah 3: Normalization
df['normalized'] = df['casefolded'].progress_apply(normalize_text)

# Langkah 4: Tokenization
df['tokens'] = df['normalized'].progress_apply(tokenize_text)

# Langkah 5: Stopword Removal
df['tokens_clean'] = df['tokens'].progress_apply(remove_stopwords)

# Langkah 6: Stemming
df['stemmed'] = df['tokens_clean'].progress_apply(stem_tokens)

print("✅ Pipeline selesai!")
print(f"Total dokumen: {len(df)}")

Memulai preprocessing pipeline...


100%|██████████| 1882/1882 [00:00<00:00, 2899.54it/s]

✅ Pipeline selesai!
Total dokumen: 1882


In [10]:
print("\n" + "="*60)
print("VERIFIKASI HASIL PREPROCESSING")
print("="*60)
print(f"\nKolom di DataFrame: {df.columns.tolist()}")
print(f"\nContoh hasil (baris pertama):")
print(f"  full_text : {df['full_text'].iloc[0][:80]}...")
print(f"  cleaned   : {df['cleaned'].iloc[0][:80]}...")
print(f"  normalized: {df['normalized'].iloc[0][:80]}...")
print(f"  tokens    : {df['tokens'].iloc[0][:10]}")
print(f"  stemmed   : {df['stemmed'].iloc[0][:10]}")

# Cek baris kosong setelah preprocessing
empty_rows = df['stemmed'].apply(lambda x: len(x) == 0).sum()
print(f"\nDokumen kosong setelah preprocessing: {empty_rows}")
if empty_rows > 0:
    df = df[df['stemmed'].apply(lambda x: len(x) > 0)].reset_index(drop=True)
    print(f"Dokumen tersisa: {len(df)}")


VERIFIKASI HASIL PREPROCESSING

Kolom di DataFrame: ['created_at', 'user_id_str', 'full_text', 'cleaned', 'casefolded', 'normalized', 'tokens', 'tokens_clean', 'stemmed']

Contoh hasil (baris pertama):
  full_text :  Mau performa mesin maksimal tanpa boros bensin? Pelajari dulu panduan campuran ...
  cleaned   : Mau performa mesin maksimal tanpa boros bensin Pelajari dulu panduan campuran et...
  normalized: mau performa mesin maksimal tanpa boros bensin pelajari dulu panduan campuran et...
  tokens    : ['mau', 'performa', 'mesin', 'maksimal', 'tanpa', 'boros', 'bensin', 'pelajari', 'dulu', 'panduan']
  stemmed   : ['performa', 'mesin', 'maksimal', 'boros', 'bensin', 'ajar', 'pandu', 'campur', 'etanol', 'bbm']

Dokumen kosong setelah preprocessing: 0


In [11]:
df.to_csv("cleanData/LDA_twitter_X_preprocessed2.csv", index=False)
print("\nFile berhasil disimpan!")


File berhasil disimpan!
